# GLIF RSNN Quickstart Tutorial

This notebook demonstrates the core patterns for training GLIF (Generalized Leaky Integrate-and-Fire) recurrent spiking neural networks using btorch.

**Key concepts covered:**
- Building a single-layer GLIF RSNN with partial adaptation
- State initialization and reset patterns (critical for btorch)
- Training loop with proper dt context
- Input/output calibration
- Checkpointing (saving/loading model + memory reset values)
- Visualization (spike raster, voltage traces)

## 1. Setup and Imports

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# btorch imports - these are the key modules for SNN work
from btorch.models import environ, functional
from btorch.models.neurons import GLIF3
from btorch.models.synapse import AlphaPSCBilleh
from btorch.models.linear import SparseConn, LearnableScale
from btorch.models.rnn import RecurrentNN
from btorch.models.init import uniform_v_

# Our glif_net modules
import sys
sys.path.append('..')

from src.model import SingleLayerGLIFRSNN
from src.loss import CombinedLoss, LossConfig
from src.checkpoint import save_checkpoint, load_checkpoint
from src.calibration import calibrate_io_scales
from src.viz import visualize_spike_raster, visualize_voltage_traces, compute_firing_rate_stats

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.8.0
CUDA available: True
Using device: cuda


## 2. Understanding btorch State Management

btorch uses **stateful neurons** - they maintain internal state (voltage, synaptic currents) across time steps. This requires special handling:

1. **`functional.init_net_state()`** - Initialize memory buffers (call once before training)
2. **`uniform_v_()`** - Set initial voltages and optionally store as reset values
3. **`functional.reset_net()`** - Reset state to stored reset values (call before each batch)
4. **`environ.context(dt=...)`** - Wrap all forward passes with timestep context

### Why this matters:
- Dynamic state buffers are **NOT** in `state_dict()` - they must be saved separately as `memories_rv`
- Forgetting to reset state causes gradients to flow across batches (wrong!)
- Forgetting dt context prevents discrete ODE solver from working

In [2]:
# Create a simple GLIF RSNN
model = SingleLayerGLIFRSNN(
    input_dim=80,        # 80 input neurons
    output_dim=10,       # 10 classes
    n_neuron=128,        # 128 recurrent neurons
    n_adapt=64,          # 64 neurons with adaptation
    asc_amp=-0.5,        # Adaptation strength
    tau_adapt=700.0,     # Adaptation time constant
    tau_mem=20.0,        # Membrane time constant
    tau_syn=5.0,         # Synaptic time constant
    v_threshold=-45.0,   # Firing threshold (mV)
    v_reset=-60.0,       # Reset voltage (mV)
    dt=1.0,              # 1ms timestep
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Adaptive neurons: {model.n_adapt} / {model.n_neuron}")

NameError: name 'build_sparse_mat' is not defined

### 2.1 State Initialization Pattern

Before training, we must:
1. Initialize memory buffers for the batch size we'll use
2. Set random voltages as reset values (deterministic reset each batch)

In [ ]:
batch_size = 32

# Step 1: Initialize memory buffers (call once at start)
functional.init_net_state(model.rnn, batch_size=batch_size, device=device)

# Step 2: Set uniform random voltages as reset values
# set_reset_value=True stores these for deterministic reset
uniform_v_(model.neuron, set_reset_value=True)

print("State initialized!")
print(f"Neuron voltage shape: {model.neuron.v.shape}")
print(f"PSC shape: {model.psc.psc.shape}")

### 2.2 Forward Pass with Proper State Management

For each batch during training:
1. **Reset state** to stored reset values
2. **Zero gradients**
3. **Forward pass** with `environ.context(dt=...)`
4. **Backward pass**
5. **Optimizer step**

In [ ]:
# Create dummy data
T = 100  # 100 timesteps
x = torch.randn(T, batch_size, 80).to(device)  # (T, batch, input_dim)

# Reset state BEFORE each forward (critical!)
functional.reset_net(model.rnn, batch_size=batch_size)

# Forward pass MUST be wrapped in environ.context(dt=...)
with environ.context(dt=1.0):
    output, states = model(x)

print(f"Output shape: {output.shape}")  # (batch, output_dim)
print(f"Spikes shape: {states['spikes'].shape}")  # (T, batch, n_neuron)
print(f"Firing rate: {states['spikes'].sum() / (T * batch_size * model.n_neuron) * 1000:.2f} Hz")

## 3. Training Loop

Putting it all together with loss computation and proper state management.

In [ ]:
def train_step(model, x, target, optimizer, loss_fn, device):
    """
    Single training step with proper btorch state management.
    
    Args:
        model: GLIF RSNN
        x: Input tensor (T, batch, input_dim)
        target: Target labels (batch,)
        optimizer: PyTorch optimizer
        loss_fn: Loss function
        device: torch.device
    """
    model.train()
    
    # Move data to device
    x = x.to(device)
    target = target.to(device)
    batch_size = x.shape[1]
    T = x.shape[0]
    
    # CRITICAL: Reset state before each batch
    functional.reset_net(model.rnn, batch_size=batch_size)
    
    # Zero gradients
    optimizer.zero_grad()
    
    # Forward pass with dt context
    with environ.context(dt=model.dt):
        output, states = model(x)
    
    # Compute loss
    loss, loss_dict = loss_fn(output, target, states, T)
    
    # Backward pass
    loss.backward()
    
    # Gradient clipping (recommended for SNNs)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # Update weights
    optimizer.step()
    
    return loss.item(), loss_dict, states

# Setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_config = LossConfig(
    ce_weight=1.0,
    voltage_weight=0.01,
    rate_weight=0.1,
    rate_target_min=2.0,
    rate_target_max=30.0,
    v_reset=-60.0,
)
loss_fn = CombinedLoss(loss_config, dt=1.0)

# Dummy training data
x_batch = torch.randn(T, batch_size, 80)
target_batch = torch.randint(0, 10, (batch_size,))

# Run one training step
loss, loss_dict, states = train_step(model, x_batch, target_batch, optimizer, loss_fn, device)

print(f"Loss: {loss:.4f}")
print(f"Loss components: {loss_dict}")

## 4. Calibration

RSNN performance depends heavily on proper input/output scaling. We use `LearnableScale` modules that can be calibrated on a few batches, then optionally frozen.

In [ ]:
# Check scales before calibration
print("Before calibration:")
print(f"  Input scale: {model.input_scale.scale.item():.4f}")
print(f"  Output scale: {model.output_scale.scale.item():.4f}")

# Create dummy dataloader
from torch.utils.data import TensorDataset, DataLoader

dummy_data = torch.randn(100, T, 80)  # 100 samples
dummy_labels = torch.randint(0, 10, (100,))
dataset = TensorDataset(dummy_data, dummy_labels)
loader = DataLoader(dataset, batch_size=32)

# Calibrate on a few batches
calibrate_io_scales(model, loader, device, num_batches=3)

print("\nAfter calibration:")
print(f"  Input scale: {model.input_scale.scale.item():.4f}")
print(f"  Output scale: {model.output_scale.scale.item():.4f}")

# Optional: Freeze scales after calibration
# model.input_scale.requires_grad_(False)
# model.output_scale.requires_grad_(False)

## 5. Checkpointing

**Critical for btorch:** Dynamic state buffers are NOT saved in `state_dict()`. We must save/restore `memories_rv` (memory reset values) separately.

In [ ]:
# Save checkpoint
checkpoint_path = Path("checkpoint_demo.pth")

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=5,
    best_acc=85.5,
    path=checkpoint_path
)

print(f"Checkpoint saved to {checkpoint_path}")

# Inspect what's in the checkpoint
checkpoint = torch.load(checkpoint_path, weights_only=False)
print(f"\nCheckpoint contents:")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Best accuracy: {checkpoint['best_acc']:.2f}%")
print(f"  Model state keys: {list(checkpoint['model_state_dict'].keys())[:5]}...")
print(f"  Memories RV keys: {list(checkpoint['memories_rv'].keys())}")

In [ ]:
# Create a new model and load checkpoint
model_new = SingleLayerGLIFRSNN(
    input_dim=80,
    output_dim=10,
    n_neuron=128,
).to(device)

# Initialize state first (required before loading)
functional.init_net_state(model_new.rnn, batch_size=32, device=device)

# New optimizer
optimizer_new = torch.optim.Adam(model_new.parameters(), lr=1e-3)

# Load checkpoint
start_epoch, best_acc = load_checkpoint(
    path=checkpoint_path,
    model=model_new,
    optimizer=optimizer_new,
    device=device
)

print(f"Loaded from epoch {start_epoch}, best_acc={best_acc:.2f}%")

# Verify scales were restored
print(f"  Input scale: {model_new.input_scale.scale.item():.4f}")
print(f"  Output scale: {model_new.output_scale.scale.item():.4f}")

In [ ]:
# Cleanup
checkpoint_path.unlink()
print("Checkpoint cleaned up")

## 6. Visualization

Plot spike rasters and voltage traces for analysis.

In [ ]:
# Generate some spikes
model.eval()

x_test = torch.randn(T, 1, 80).to(device)  # Single sample

with torch.no_grad():
    with environ.context(dt=1.0):
        functional.reset_net(model.rnn, batch_size=1)
        output, states = model(x_test)

spikes = states['spikes'].cpu()[:, 0, :]  # (T, n_neuron)
print(f"Spikes shape: {spikes.shape}")
print(f"Total spikes: {spikes.sum().item():.0f}")

# Compute stats
stats = compute_firing_rate_stats(states['spikes'], dt=1.0)
print(f"\nFiring rate stats:")
for k, v in stats.items():
    print(f"  {k}: {v:.2f}")

In [ ]:
# Plot spike raster
fig, ax = plt.subplots(figsize=(12, 6))

# Select first 50 neurons for visibility
n_show = min(50, model.n_neuron)
spikes_np = spikes[:, :n_show].numpy()

# Create raster plot
for neuron_idx in range(n_show):
    spike_times = np.where(spikes_np[:, neuron_idx] > 0)[0]
    ax.scatter(spike_times, [neuron_idx] * len(spike_times), 
               c='black', s=10, marker='|')

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron Index')
ax.set_title('Spike Raster Plot')
ax.set_xlim(0, T)
ax.set_ylim(-0.5, n_show - 0.5)
plt.tight_layout()
plt.show()

**Key Takeaways:**

1. **Always use `environ.context(dt=...)`** - Wrap all forward passes
2. **Always call `functional.reset_net()`** - Before each batch
3. **Save/restore `memories_rv`** - Dynamic states aren't in state_dict
4. **Use `LearnableScale`** - For input/output calibration
5. **Clip gradients** - SNNs benefit from gradient clipping (max_norm=1.0)